# Projet Deep Learning — Reconnaissance d'expressions faciales

L'idée : apprendre à un réseau de neurones à reconnaître l'émotion d'un visage (joie, colère, tristesse...).
On avance étape par étape : d'abord les données, puis un modèle simple, puis un CNN, et on améliore au fur et à mesure.

## Partie 1 — Les données

### 1.1 Installation et imports

Sur Colab, TensorFlow est déjà installé. Il nous manque juste `kagglehub` pour récupérer le dataset directement depuis Kaggle.

In [ ]:
%pip install -q kagglehub

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
import kagglehub

print("TensorFlow :", tf.__version__)

### 1.2 Le dataset choisi : FER-2013

On part sur **FER-2013**, le dataset de référence pour ce genre de projet :
- **Source** : challenge Kaggle *"Challenges in Representation Learning: Facial Expression Recognition"* (ICML 2013), créé par Pierre-Luc Carrier et Aaron Courville. On utilise la version en images de [msambare sur Kaggle](https://www.kaggle.com/datasets/msambare/fer2013).
- **Licence** : usage pour la recherche / l'éducation (les images viennent de Google Images).
- **Contenu** : des visages déjà recadrés, en **48×48 pixels**, en **niveaux de gris**, au format **JPG**.
- **7 classes** : angry, disgust, fear, happy, neutral, sad, surprise.

Le dataset est déjà séparé en deux dossiers `train/` et `test/`, avec un sous-dossier par émotion. Pratique : le nom du dossier, c'est le label !

In [ ]:
# Téléchargement (la première fois ça prend quelques secondes, ensuite c'est en cache)
data_dir = kagglehub.dataset_download("msambare/fer2013")
train_dir = os.path.join(data_dir, "train")
test_dir = os.path.join(data_dir, "test")

print("Dataset téléchargé dans :", data_dir)
print("Contenu :", os.listdir(data_dir))

### 1.3 Premier coup d'œil

On compte combien d'images il y a dans chaque classe, pour le train et le test.

In [ ]:
classes = sorted(os.listdir(train_dir))
print("Classes :", classes)

# On compte les fichiers dans chaque dossier
comptes = pd.DataFrame({
    "train": [len(os.listdir(os.path.join(train_dir, c))) for c in classes],
    "test": [len(os.listdir(os.path.join(test_dir, c))) for c in classes],
}, index=classes)
comptes.loc["TOTAL"] = comptes.sum()
comptes

Et on ouvre une image au hasard pour vérifier sa taille et son format :

In [ ]:
dossier = os.path.join(train_dir, "happy")
fichier = os.listdir(dossier)[0]
img = Image.open(os.path.join(dossier, fichier))

print("Fichier :", fichier)
print("Taille :", img.size, "| Mode :", img.mode, "(L = niveaux de gris)")

plt.imshow(img, cmap="gray")
plt.title("happy")
plt.axis("off")
plt.show()

### 1.4 Toutes les images ont-elles le même format ?

On a regardé une seule image, mais rien ne garantit que toutes sont pareilles. On vérifie sur tout le dataset (PIL lit juste l'en-tête du fichier, donc ça va vite).

In [ ]:
tailles = set()
modes = set()
extensions = set()

for dossier in [train_dir, test_dir]:
    for c in classes:
        for f in os.listdir(os.path.join(dossier, c)):
            img = Image.open(os.path.join(dossier, c, f))
            tailles.add(img.size)
            modes.add(img.mode)
            extensions.add(os.path.splitext(f)[1])

print("Tailles trouvées :", tailles)
print("Modes trouvés :", modes)
print("Extensions trouvées :", extensions)

### 1.5 Répartition des classes

On trace le nombre d'images par émotion. Si les barres sont très différentes, le dataset est **déséquilibré** : le modèle risque de favoriser les classes qui ont beaucoup d'exemples.

In [ ]:
comptes_classes = comptes.drop("TOTAL")

comptes_classes.plot(kind="bar", figsize=(9, 4))
plt.title("Nombre d'images par classe")
plt.ylabel("Nombre d'images")
plt.xticks(rotation=0)
plt.show()

# En pourcentage, c'est plus parlant
pourcentages = (comptes_classes["train"] / comptes_classes["train"].sum() * 100).round(1)
print("Part de chaque classe dans le train (%) :")
print(pourcentages)
print()
print("Ratio classe la plus fréquente / la moins fréquente :",
      round(comptes_classes["train"].max() / comptes_classes["train"].min(), 1))

**Ce qu'on remarque :**
- `happy` écrase tout le monde (~25 % du train).
- `disgust` est quasi absente (~1.5 %) : environ 16 fois moins d'images que `happy`.
- Le test a la même répartition que le train, donc l'accuracy sur le test sera elle aussi tirée par `happy`.

Conséquence : un modèle qui répondrait "happy" à chaque fois aurait déjà ~25 % d'accuracy sans rien apprendre. C'est pour ça qu'on regardera la **matrice de confusion** et les scores **par classe** en partie 5, et pas juste l'accuracy globale.

### 1.6 À quoi ressemblent les visages ?

On affiche quelques images au hasard pour chaque émotion (une ligne = une classe).

In [ ]:
nb_par_classe = 8
np.random.seed(0)

fig, axes = plt.subplots(len(classes), nb_par_classe, figsize=(nb_par_classe * 1.3, len(classes) * 1.5))

for i, c in enumerate(classes):
    fichiers = os.listdir(os.path.join(train_dir, c))
    choix = np.random.choice(fichiers, nb_par_classe, replace=False)
    for j, f in enumerate(choix):
        img = Image.open(os.path.join(train_dir, c, f))
        axes[i, j].imshow(img, cmap="gray")
        axes[i, j].axis("off")
    axes[i, 0].set_title(c, loc="left", fontsize=11)

plt.tight_layout()
plt.show()

**Ce qu'on remarque :**
- Les visages sont recadrés et à peu près centrés, mais l'angle, l'éclairage et l'âge des personnes varient beaucoup.
- Certaines images sont floues, partiellement cachées (mains, lunettes de soleil), ou ont un filigrane par-dessus : FER-2013 est connu pour être un peu "bruité".
- Certaines émotions se ressemblent beaucoup (`fear` / `surprise`, `sad` / `neutral`, `angry` / `disgust`) : on peut s'attendre à ce que le modèle les confonde.

### 1.7 Préparation des données

Un réseau de neurones ne lit pas des fichiers JPG : il veut des **tableaux de nombres**. On va donc :
1. **Charger** toutes les images dans des tableaux numpy (`X` = les images, `y` = les labels).
2. **Redimensionner** : pas besoin ici, on a vérifié que tout est déjà en 48×48.
3. **Normaliser** les pixels.
4. **Encoder** les labels.
5. **Découper** en train / validation / test.

#### Chargement

Pour chaque image, on garde ses pixels dans `X` et le numéro de sa classe dans `y` (0 = angry, 1 = disgust, ..., 6 = surprise). Le numéro vient simplement de la position du dossier dans la liste `classes`.

In [ ]:
X_train_full, y_train_full = [], []
X_test, y_test = [], []

for i, c in enumerate(classes):
    for f in os.listdir(os.path.join(train_dir, c)):
        X_train_full.append(np.array(Image.open(os.path.join(train_dir, c, f))))
        y_train_full.append(i)
    for f in os.listdir(os.path.join(test_dir, c)):
        X_test.append(np.array(Image.open(os.path.join(test_dir, c, f))))
        y_test.append(i)

X_train_full = np.array(X_train_full)
y_train_full = np.array(y_train_full)
X_test = np.array(X_test)
y_test = np.array(y_test)

print("X_train_full :", X_train_full.shape, X_train_full.dtype)
print("y_train_full :", y_train_full.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

On a 28 709 images de 48×48 pour le train. Keras attend aussi une dimension pour les **canaux** (1 en niveaux de gris, 3 en couleur), donc on passe de `(N, 48, 48)` à `(N, 48, 48, 1)`.

In [ ]:
X_train_full = X_train_full.reshape(-1, 48, 48, 1)
X_test = X_test.reshape(-1, 48, 48, 1)

print("X_train_full :", X_train_full.shape)
print("X_test :", X_test.shape)

#### Normalisation

Les pixels vont de 0 (noir) à 255 (blanc). On les divise par 255 pour les ramener entre **0 et 1**.

Pourquoi ? Avec des valeurs trop grandes, les sommes pondérées dans les neurones explosent et les gradients aussi : l'entraînement devient instable et lent. Avec des entrées entre 0 et 1, tout reste à la même échelle et la descente de gradient se passe beaucoup mieux.

In [ ]:
# On garde une copie des valeurs brutes juste pour comparer
pixels_avant = X_train_full[:1000].flatten()

X_train_full = X_train_full.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

pixels_apres = X_train_full[:1000].flatten()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(pixels_avant, bins=50)
axes[0].set_title("Pixels avant normalisation (0 → 255)")
axes[1].hist(pixels_apres, bins=50, color="tab:orange")
axes[1].set_title("Pixels après normalisation (0 → 1)")
plt.show()

print("Min / max après :", X_train_full.min(), "/", X_train_full.max())

La forme de l'histogramme ne change pas du tout, seule l'échelle de l'axe x change : on n'a perdu aucune information.

#### Découpage train / validation / test

On a déjà un jeu de **test** fourni par le dataset. On n'y touche plus jusqu'à l'évaluation finale : si on s'en servait pour régler le modèle, le score final serait trop optimiste.

Pour suivre l'entraînement et comparer nos modèles, on a besoin d'un jeu de **validation**. On le prend dans le train : 15 % des images, tirées au hasard.

On utilise `stratify=y_train_full` pour garder **la même répartition des classes** dans le train et la validation. Sans ça, avec seulement 436 images `disgust`, on pourrait se retrouver avec très peu de `disgust` en validation.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15,
    stratify=y_train_full,
    random_state=42,
)

print("Train      :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

On vérifie que la stratification a bien marché (les pourcentages doivent être quasi identiques) :

In [ ]:
repartition = pd.DataFrame({
    "train (%)": np.bincount(y_train) / len(y_train) * 100,
    "validation (%)": np.bincount(y_val) / len(y_val) * 100,
    "test (%)": np.bincount(y_test) / len(y_test) * 100,
}, index=classes).round(1)
repartition

#### Encodage des labels (one-hot)

Pour l'instant un label est un simple numéro, par exemple `3` pour happy. Mais notre réseau va sortir **7 probabilités** (une par classe, grâce au Softmax). Pour pouvoir comparer la sortie avec la vraie réponse, on transforme le label en vecteur de 7 cases avec un 1 à la bonne place :

`3` → `[0, 0, 0, 1, 0, 0, 0]`

C'est le **one-hot encoding**. Ça évite aussi que le réseau croie qu'il y a un ordre entre les classes (`surprise` = 6 n'est pas "plus grand" que `angry` = 0).

On garde aussi les versions en numéros (`y_train`, `y_val`, `y_test`), elles seront pratiques pour la matrice de confusion.

In [ ]:
from tensorflow.keras.utils import to_categorical

y_train_oh = to_categorical(y_train, num_classes=len(classes))
y_val_oh = to_categorical(y_val, num_classes=len(classes))
y_test_oh = to_categorical(y_test, num_classes=len(classes))

print("y_train :", y_train.shape, "→ y_train_oh :", y_train_oh.shape)

### 1.8 Vérification finale

Avant de passer aux modèles, on affiche quelques images **telles que le réseau va les recevoir** (déjà normalisées), avec leur label en numéro, en one-hot et en nom. Si tout colle, la préparation est bonne.

In [ ]:
np.random.seed(1)
indices = np.random.choice(len(X_train), 6, replace=False)

fig, axes = plt.subplots(1, 6, figsize=(15, 3.5))
for ax, idx in zip(axes, indices):
    ax.imshow(X_train[idx].squeeze(), cmap="gray")
    ax.set_title(f"{classes[y_train[idx]]} ({y_train[idx]})\n{y_train_oh[idx].astype(int)}", fontsize=9)
    ax.axis("off")
plt.show()

**Récap de ce qu'on a en sortie de la partie 1 :**

| Jeu | Variable | Forme | Rôle |
|---|---|---|---|
| Train | `X_train`, `y_train_oh` | (24 402, 48, 48, 1) | apprendre les poids |
| Validation | `X_val`, `y_val_oh` | (4 307, 48, 48, 1) | suivre l'entraînement, comparer les modèles |
| Test | `X_test`, `y_test_oh` | (7 178, 48, 48, 1) | évaluation finale uniquement |

## Partie 2 — Modèle de référence (réseau dense)

Avant de faire un CNN, on construit un réseau "classique" tout simple. Il ne sera pas très bon, et c'est normal : il sert de **point de comparaison**. Si notre CNN ne fait pas mieux que lui, c'est qu'il y a un problème.

Architecture : **Image → Flatten → Dense(128) + ReLU → Dense(7) + Softmax**

### 2.1 Comment ça marche

**L'image en entrée.** Une couche Dense ne comprend pas les images en 2D : elle veut une simple liste de nombres. La couche `Flatten` "déroule" l'image 48×48 en un vecteur de 48 × 48 = **2 304 valeurs**. On perd au passage l'information de voisinage entre pixels (le réseau ne sait plus quel pixel est à côté de quel autre) : c'est justement ce que le CNN va corriger.

**Poids et biais.** Chaque neurone d'une couche Dense est relié à toutes les entrées. Il calcule une somme pondérée : `z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b`. Les **poids** `w` disent quelle importance donner à chaque entrée, le **biais** `b` décale le résultat. Ce sont eux que le réseau apprend.

**Fonction d'activation.** On applique ensuite une fonction non linéaire à `z`. On utilise **ReLU** : `ReLU(z) = max(0, z)`. Sans activation, empiler des couches ne servirait à rien : une suite d'opérations linéaires reste linéaire, et le réseau ne pourrait apprendre que des frontières toutes droites.

**Propagation avant.** L'image passe couche par couche : Flatten → Dense(128) + ReLU → Dense(7) + Softmax. À la fin on obtient 7 probabilités.

**Sortie multiclasse.** La dernière couche a **7 neurones** (un par émotion) avec **Softmax**, qui transforme les 7 valeurs en probabilités positives dont la somme fait 1. La classe prédite est celle qui a la plus grande probabilité.

**Fonction de perte.** Pour mesurer à quel point le réseau se trompe, on utilise la **categorical cross-entropy** : `loss = -log(p_vraie_classe)`. Si le réseau donne 0.9 à la bonne classe, la loss vaut 0.1 (faible). S'il donne 0.05, elle vaut 3 (forte). Elle punit surtout les erreurs faites avec confiance.

**Rétropropagation.** Une fois la loss calculée, on calcule pour chaque poids son **gradient** : de combien la loss change si on bouge ce poids un tout petit peu. On part de la sortie et on remonte couche par couche vers l'entrée grâce à la règle de dérivation en chaîne : c'est la *backpropagation*.

**Descente de gradient.** On bouge ensuite chaque poids un peu dans le sens qui fait **baisser** la loss : `w ← w - learning_rate × gradient`. On répète ça sur des petits paquets d'images (les *batchs*) pendant plusieurs *époques* (une époque = un passage sur tout le train). On utilise **Adam**, une version améliorée de la descente de gradient qui adapte le pas pour chaque poids.

### 2.2 Construction du modèle

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# On fixe la graine aléatoire pour avoir des résultats reproductibles
keras.utils.set_random_seed(42)

baseline = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="softmax"),
])

baseline.summary()

Le compte des paramètres :
- `Flatten` : 0 paramètre, il ne fait que réorganiser les pixels.
- `Dense(128)` : 2 304 entrées × 128 neurones + 128 biais = **295 040**.
- `Dense(7)` : 128 × 7 + 7 biais = **903**.

Presque tous les paramètres sont dans la première couche, parce que chaque neurone est connecté à chacun des 2 304 pixels.

### 2.3 Entraînement

On entraîne 20 époques avec Adam, des batchs de 64 images, et on suit la loss et l'accuracy sur la validation à chaque époque.

In [ ]:
baseline.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_baseline = baseline.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=20,
    batch_size=64,
    verbose=2,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_baseline.history["loss"], label="train")
axes[0].plot(history_baseline.history["val_loss"], label="validation")
axes[0].set_title("Loss — modèle dense")
axes[0].set_xlabel("Époque")
axes[0].legend()

axes[1].plot(history_baseline.history["accuracy"], label="train")
axes[1].plot(history_baseline.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy — modèle dense")
axes[1].set_xlabel("Époque")
axes[1].legend()

plt.show()

print("Accuracy validation finale :", round(history_baseline.history["val_accuracy"][-1], 3))
print("Pour rappel, toujours répondre 'happy' donnerait ~0.25")

**Ce qu'on retient :**
- Le modèle dense apprend un peu : ~37 % en validation, mieux que les ~25 % qu'on aurait en répondant toujours "happy". Mais ça reste faible.
- Il plafonne vite : la validation n'avance plus après ~8 époques, alors que le train continue de s'améliorer. Les deux courbes de loss s'écartent : c'est un début de **surapprentissage** (le modèle commence à apprendre le train par cœur au lieu de généraliser).
- Comme il voit l'image comme une liste de pixels sans structure, il ne peut pas repérer une bouche ou des sourcils où qu'ils soient dans l'image.

C'est notre score à battre avec le CNN.

> On ne l'évalue pas encore sur le test : on comparera tous les modèles sur le test en une seule fois, à la fin.

## Partie 3 — Construction d'un CNN

### 3.1 Les briques d'un CNN

**Filtres et convolution.** Un filtre est une petite grille de poids (ici 3×3). On le fait glisser sur toute l'image : à chaque position, on multiplie les 9 pixels sous le filtre par les 9 poids et on additionne. Si la zone ressemble au motif du filtre (un bord horizontal, un coin...), le résultat est grand. Les poids des filtres ne sont pas choisis à la main : ils sont **appris** pendant l'entraînement, comme les poids d'une couche Dense.

**Feature maps.** Le résultat d'un filtre passé sur toute l'image est une nouvelle "image" qui montre où le motif a été trouvé : c'est une **feature map**. Avec 32 filtres, on obtient 32 feature maps.

**Taille du kernel.** C'est la taille du filtre. On prend **3×3** : assez petit pour peu de paramètres, et en empilant les couches, le réseau finit quand même par voir de grandes zones de l'image.

**Stride.** Le pas avec lequel le filtre se déplace. On garde **stride = 1** (le filtre avance pixel par pixel) et on laisse le pooling réduire la taille.

**Padding.** Sans rien faire, un filtre 3×3 ne peut pas se centrer sur les pixels du bord, et l'image rétrécit de 2 pixels à chaque convolution. Avec `padding="same"`, on ajoute une bordure de zéros autour de l'image pour **garder la même taille** en sortie.

**ReLU.** Comme pour le modèle dense, on applique `max(0, z)` après chaque convolution pour ajouter de la non-linéarité. Ça garde seulement les endroits où le motif est présent (valeurs positives).

**Pooling.** Le **MaxPooling 2×2** découpe chaque feature map en carrés de 2×2 et garde juste le max de chaque carré. La taille est **divisée par 2**. Ça réduit le calcul et rend le réseau un peu insensible aux petits décalages (si la bouche bouge d'un pixel, le max reste à peu près le même).

**Flatten + Dense.** Une fois les caractéristiques extraites, on "déroule" les feature maps en vecteur et on les donne à des couches Dense classiques qui décident de l'émotion.

**Couche de sortie.** Comme pour le modèle dense : **Dense(7) + Softmax** pour avoir une probabilité par émotion.

### 3.2 Notre architecture

On empile **3 blocs Conv + Pool** avec de plus en plus de filtres (32 → 64 → 128) :
- les premières couches repèrent des choses simples (bords, contrastes) ;
- les suivantes combinent ces motifs en formes plus complexes (yeux, coins de la bouche, sourcils froncés).

On double le nombre de filtres à chaque fois que le pooling divise la taille par 2, pour ne pas perdre trop d'information. 3 blocs, c'est un bon compromis pour des images de 48×48 : après 3 poolings, on est à 6×6, en dessous ça deviendrait trop petit.

In [ ]:
keras.utils.set_random_seed(42)

cnn = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),

    # Bloc 1 : motifs simples (bords, contrastes)
    layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    # Bloc 2 : motifs un peu plus complexes
    layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    # Bloc 3 : parties du visage
    layers.Conv2D(128, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=2),

    # Classification
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="softmax"),
])

cnn.summary()

### 3.3 Évolution des dimensions et nombre de paramètres

Pour une Conv2D, le nombre de paramètres vaut : `(taille kernel × taille kernel × nb canaux en entrée + 1) × nb filtres` (le +1 c'est le biais de chaque filtre).

| Couche | Sortie | Calcul des paramètres | Paramètres |
|---|---|---|---|
| Entrée | 48 × 48 × 1 | — | 0 |
| Conv2D 32 filtres 3×3 | 48 × 48 × 32 | (3×3×1 + 1) × 32 | 320 |
| MaxPooling 2×2 | 24 × 24 × 32 | — | 0 |
| Conv2D 64 filtres 3×3 | 24 × 24 × 64 | (3×3×32 + 1) × 64 | 18 496 |
| MaxPooling 2×2 | 12 × 12 × 64 | — | 0 |
| Conv2D 128 filtres 3×3 | 12 × 12 × 128 | (3×3×64 + 1) × 128 | 73 856 |
| MaxPooling 2×2 | 6 × 6 × 128 | — | 0 |
| Flatten | 4 608 | — | 0 |
| Dense 128 | 128 | 4 608 × 128 + 128 | 589 952 |
| Dense 7 (Softmax) | 7 | 128 × 7 + 7 | 903 |
| **Total** | | | **683 527** |

**Ce qu'on remarque :**
- Grâce au `padding="same"`, la taille ne bouge pas dans les Conv2D, seul le nombre de canaux change. C'est le pooling qui divise la taille par 2.
- Les couches de convolution ont **peu de paramètres** (un filtre 3×3 est réutilisé partout sur l'image), alors que la première Dense en a à elle seule **86 %**. C'est une piste pour plus tard si le modèle sur-apprend.

### 3.4 À quoi ressemblent les feature maps ?

On fait passer un visage dans la première couche de convolution et on affiche les 32 feature maps produites.

Attention : le réseau n'est **pas encore entraîné**, ses filtres sont donc aléatoires. On voit déjà qu'une convolution fait ressortir des zones différentes de l'image (bords, zones claires ou sombres). On refera l'exercice après l'entraînement pour comparer.

In [ ]:
# Un mini-modèle qui s'arrête après la première couche de convolution
extracteur = keras.Model(inputs=cnn.inputs, outputs=cnn.layers[0].output)

visage = X_train[0:1]  # on garde la dimension batch : (1, 48, 48, 1)
feature_maps = extracteur.predict(visage, verbose=0)
print("Forme des feature maps :", feature_maps.shape)

plt.imshow(visage[0].squeeze(), cmap="gray")
plt.title(f"Image d'entrée : {classes[y_train[0]]}")
plt.axis("off")
plt.show()

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(feature_maps[0, :, :, i], cmap="viridis")
    ax.axis("off")
plt.suptitle("Les 32 feature maps de la 1re couche (filtres aléatoires, avant entraînement)")
plt.show()

**Ce qu'on remarque :**
- Chaque filtre fait ressortir quelque chose de différent : certains mettent en avant les contours (yeux, bouche), d'autres les zones claires ou sombres.
- Beaucoup de feature maps sont presque entièrement noires : c'est l'effet du **ReLU**, qui met à 0 toutes les valeurs négatives. Avec des filtres aléatoires, beaucoup ne "réagissent" à rien sur cette image.
- La taille reste bien 48×48 grâce au `padding="same"`.